# LeetCode 2120 — Execution of All Suffix Instructions Staying in a Grid bằng Python

Link đề: https://leetcode.com/problems/execution-of-all-suffix-instructions-staying-in-a-grid/ · Độ khó: **Medium** · Chủ đề: String, Simulation, Prefix Sum, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def executeInstructions(self, n: int, startPos: List[int], s: str) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Mô phỏng chuyển động trên lưới bằng **bảng hướng** (`dict` ký tự → `(dr, dc)`) và kiểm tra biên gọn gàng.
2. Biết khi nào **mô phỏng O(m²)** là đủ (m ≤ 500) và vẫn hiểu cách tối ưu khi m lớn.
3. Dùng **tổng tiền tố của độ dời** (prefix displacement): vị trí sau khi chạy `s[i..k-1]` = `start + P[k] − P[i]`.
4. Nhận ra tính chất **"bước ±1 không nhảy cóc"** ⇒ lần đầu ra khỏi lưới = lần đầu toạ độ chạm đúng −1 hoặc n ⇒ tra cứu bằng **hash map quét từ phải** cho lời giải O(m).
5. Biết thêm cách **sparse table + tìm nhị phân** (O(m log m)) như một cầu nối giữa hai cách trên.

## 2. Đề bài & ví dụ

Có lưới `n × n`, ô trái-trên là `(0, 0)`, ô phải-dưới là `(n−1, n−1)`. Robot đứng ở `startPos = [startrow, startcol]`.

Chuỗi lệnh `s` dài `m`, mỗi ký tự là `'L'` (trái: cột −1), `'R'` (phải: cột +1), `'U'` (lên: hàng −1), `'D'` (xuống: hàng +1).

Robot có thể **bắt đầu** từ lệnh thứ `i` bất kỳ, rồi chạy lần lượt tới cuối chuỗi. Nó **dừng** khi lệnh kế tiếp sẽ đưa nó ra khỏi lưới, hoặc khi hết lệnh. Mỗi lần bắt đầu là một lần chạy độc lập, luôn xuất phát từ `startPos`.

Trả về mảng `answer` dài `m`, `answer[i]` = số lệnh robot thực hiện được nếu bắt đầu từ lệnh `i`.

**Ví dụ 1:** `n = 3, startPos = [0,1], s = "RRDDLU"` → `[1,5,4,3,1,0]`.

**Ví dụ 2:** `n = 2, startPos = [1,1], s = "LURD"` → `[4,1,0,0]`.

**Ví dụ 3:** `n = 1, startPos = [0,0], s = "LRUD"` → `[0,0,0,0]` (lưới 1 ô, đi đâu cũng ra ngoài).

**Ràng buộc:** `m == s.length`, `1 ≤ n, m ≤ 500`, `0 ≤ startrow, startcol < n`, `s` chỉ gồm `L R U D`.

In [1]:
from typing import List
import random, time

examples = [
    (3, [0, 1], "RRDDLU", [1, 5, 4, 3, 1, 0]),
    (2, [1, 1], "LURD", [4, 1, 0, 0]),
    (1, [0, 0], "LRUD", [0, 0, 0, 0]),
]
def show_grid(n, pos):
    for r in range(n):
        print(" ".join("R" if [r, c] == pos else "." for c in range(n)))
for n, st, s, exp in examples:
    print(f"n={n}, start={st}, s={s!r} -> kỳ vọng {exp}")
    show_grid(n, st); print()

n=3, start=[0, 1], s='RRDDLU' -> kỳ vọng [1, 5, 4, 3, 1, 0]
. R .
. . .
. . .

n=2, start=[1, 1], s='LURD' -> kỳ vọng [4, 1, 0, 0]
. .
. R

n=1, start=[0, 0], s='LRUD' -> kỳ vọng [0, 0, 0, 0]
R



## 3. Kiến thức nền

### 3.1 Bảng hướng bằng `dict`

Thay vì chuỗi `if/elif` dài, lưu độ dời của mỗi lệnh. Lưu ý quy ước: **hàng** tăng khi đi **xuống** (`'D'`), giảm khi đi lên (`'U'`).

In [2]:
DIRS = {"L": (0, -1), "R": (0, 1), "U": (-1, 0), "D": (1, 0)}
r, c = 0, 1
for ch in "RD":
    dr, dc = DIRS[ch]
    r, c = r + dr, c + dc
    print(f"sau '{ch}': ({r}, {c})")

sau 'R': (0, 2)
sau 'D': (1, 2)


### 3.2 Kiểm tra trong lưới bằng so sánh chuỗi `0 <= x < n`

Python cho phép viết so sánh liên tiếp: `0 <= r < n` tương đương `0 <= r and r < n`.

In [3]:
def inside(n, r, c):
    return 0 <= r < n and 0 <= c < n
print(inside(3, 0, 2), inside(3, -1, 0), inside(3, 2, 3))

True False False


### 3.3 Tổng tiền tố (prefix sum) của độ dời

Gọi `PR[k]`, `PC[k]` là tổng độ dời hàng/cột sau `k` lệnh đầu (`PR[0] = PC[0] = 0`). Khi bắt đầu từ lệnh `i` và đã chạy xong lệnh `k − 1` (k > i), robot ở:

`(startrow + PR[k] − PR[i], startcol + PC[k] − PC[i])`.

Như vậy mọi lần chạy đều được mô tả bằng **hiệu** hai phần tử của cùng một mảng tiền tố.

In [4]:
def prefix(s):
    PR, PC = [0], [0]
    for ch in s:
        dr, dc = DIRS[ch]
        PR.append(PR[-1] + dr); PC.append(PC[-1] + dc)
    return PR, PC

PR, PC = prefix("RRDDLU")
print("PR =", PR)
print("PC =", PC)
i, k = 1, 4          # bắt đầu từ lệnh 1, chạy xong lệnh 3
print("vị trí:", (0 + PR[k] - PR[i], 1 + PC[k] - PC[i]))

PR = [0, 0, 0, 1, 2, 2, 1]
PC = [0, 1, 2, 2, 2, 1, 1]
vị trí: (2, 2)


### 3.4 Duyệt ngược và `dict` "lần xuất hiện gần nhất"

Khi quét `i` từ cuối về đầu và ghi `last[value] = index`, giá trị ghi sau cùng là **chỉ số nhỏ nhất** đã gặp ⇒ `last[v]` là lần xuất hiện **gần nhất bên phải** của `v` so với vị trí hiện tại.

In [5]:
a = [3, 1, 3, 2, 1, 3]
last = {}
for i in range(len(a) - 1, -1, -1):
    last[a[i]] = i
    print(f"i={i}, a[i]={a[i]}, lần gần nhất của mỗi giá trị bên phải-hoặc-tại i: {dict(sorted(last.items()))}")

i=5, a[i]=3, lần gần nhất của mỗi giá trị bên phải-hoặc-tại i: {3: 5}
i=4, a[i]=1, lần gần nhất của mỗi giá trị bên phải-hoặc-tại i: {1: 4, 3: 5}
i=3, a[i]=2, lần gần nhất của mỗi giá trị bên phải-hoặc-tại i: {1: 4, 2: 3, 3: 5}
i=2, a[i]=3, lần gần nhất của mỗi giá trị bên phải-hoặc-tại i: {1: 4, 2: 3, 3: 2}
i=1, a[i]=1, lần gần nhất của mỗi giá trị bên phải-hoặc-tại i: {1: 1, 2: 3, 3: 2}
i=0, a[i]=3, lần gần nhất của mỗi giá trị bên phải-hoặc-tại i: {1: 1, 2: 3, 3: 0}


## 4. Ý tưởng nền tảng / trực giác

Làm tay ví dụ 1 (`n = 3`, start `(0, 1)`, `s = "RRDDLU"`):

| bắt đầu i | hậu tố | các bước | dừng vì | answer |
|---|---|---|---|---|
| 0 | RRDDLU | R→(0,2), R→(0,3) ra ngoài | cột = 3 | 1 |
| 1 | RDDLU | R(0,2) D(1,2) D(2,2) L(2,1) U(1,1) | hết lệnh | 5 |
| 2 | DDLU | D(1,1) D(2,1) L(2,0) U(1,0) | hết lệnh | 4 |
| 3 | DLU | D(1,1) L(1,0) U(0,0) | hết lệnh | 3 |
| 4 | LU | L(0,0), U→(−1,0) ra ngoài | hàng = −1 | 1 |
| 5 | U | U→(−1,1) ra ngoài | hàng = −1 | 0 |

Quan sát:
1. Mỗi lần chạy **độc lập**, luôn xuất phát từ `startPos` — không dùng lại vị trí của lần chạy trước.
2. Mỗi lệnh chỉ đổi **một** toạ độ đúng **1 đơn vị** ⇒ robot không thể "nhảy qua" biên: muốn ra ngoài theo hàng thì hàng phải bằng đúng `−1` hoặc `n`.
3. "Ra ngoài lần đầu" ⇔ "lần đầu toạ độ chạm −1 hoặc n" ⇒ có thể chuyển thành bài **tìm lần xuất hiện đầu tiên của một giá trị** trong mảng tiền tố.

In [6]:
def simulate_verbose(n, start, s, i):
    r, c = start; steps = 0; path = []
    for ch in s[i:]:
        dr, dc = DIRS[ch]
        if not (0 <= r + dr < n and 0 <= c + dc < n):
            path.append(f"{ch}->({r+dr},{c+dc}) RA NGOÀI"); break
        r, c = r + dr, c + dc; steps += 1; path.append(f"{ch}({r},{c})")
    return steps, path

for i in range(6):
    st, p = simulate_verbose(3, [0, 1], "RRDDLU", i)
    print(f"i={i}: {st}  | {' '.join(p)}")

i=0: 1  | R(0,2) R->(0,3) RA NGOÀI
i=1: 5  | R(0,2) D(1,2) D(2,2) L(2,1) U(1,1)
i=2: 4  | D(1,1) D(2,1) L(2,0) U(1,0)
i=3: 3  | D(1,1) L(1,0) U(0,0)
i=4: 1  | L(0,0) U->(-1,0) RA NGOÀI
i=5: 0  | U->(-1,1) RA NGOÀI


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng hậu tố

**Ý tưởng:** với mỗi `i`, đặt robot về `startPos`, chạy từ lệnh `i` tới khi ra ngoài hoặc hết lệnh, đếm số bước. Đây cũng là lời giải doocs và **được chấp nhận** vì `m ≤ 500` (≤ 125 nghìn bước).

In [7]:
def simulate_all(n, startPos, s):
    m = len(s)
    ans = []
    for i in range(m):
        r, c = startPos
        cnt = 0
        for j in range(i, m):
            dr, dc = DIRS[s[j]]
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n:
                r, c, cnt = nr, nc, cnt + 1
            else:
                break
        ans.append(cnt)
    return ans

for n, st, s, exp in examples:
    got = simulate_all(n, st, s)
    print(got, "OK" if got == exp else "SAI")

[1, 5, 4, 3, 1, 0] OK
[4, 1, 0, 0] OK
[0, 0, 0, 0] OK


**Độ phức tạp:** O(m²) thời gian worst-case (khi robot hầu như không bao giờ ra ngoài, ví dụ lưới lớn và lệnh qua lại `"LRLR..."`), O(m) bộ nhớ cho đáp án.

**Nút thắt:** mỗi hậu tố được chạy lại từ đầu dù các hậu tố chia sẻ cùng một dãy lệnh. Với `m = 500` thì không sao; nhưng nếu đề cho `m = 10⁵` thì O(m²) = 10¹⁰ bước — quá chậm. Hai cách tiếp theo dùng cùng một mảng tiền tố cho mọi `i`.

In [8]:
for m in [500, 2000]:
    s = "LR" * (m // 2)
    t = time.perf_counter(); simulate_all(m, [m // 2, m // 2], s); dt = time.perf_counter() - t
    print(f"m={m}: {dt*1000:.1f} ms")

m=500: 11.8 ms


m=2000: 236.4 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tiền tố):** vị trí sau khi chạy `s[i..k−1]` là `start + P[k] − P[i]`. Robot ra khỏi lưới ở bước `k` (tức lệnh `k − 1` là lệnh thất bại) khi:
- `PR[k] − PR[i] = −startrow − 1` (hàng −1), hoặc `= n − startrow` (hàng n), hoặc
- `PC[k] − PC[i] = −startcol − 1`, hoặc `= n − startcol`.

**Insight 2 (không nhảy cóc):** vì mỗi bước đổi đúng một toạ độ đúng 1 đơn vị, toạ độ hàng chỉ có thể rời đoạn `[0, n−1]` bằng cách đi qua đúng giá trị −1 hoặc n. Nên **lần thất bại đầu tiên** là `k* = min k > i` sao cho `PR[k]` bằng một trong hai giá trị mục tiêu `PR[i] − startrow − 1`, `PR[i] + n − startrow`, hoặc `PC[k]` bằng một trong hai mục tiêu tương ứng.

Quy tắc:
- **Nếu** tồn tại `k*` **thì** `answer[i] = k* − i − 1` (chạy được các lệnh `i..k*−2`, lệnh `k*−1` thất bại).
- **Nếu** không tồn tại **thì** `answer[i] = m − i` (chạy hết).

**Insight 3 (quét từ phải + dict):** "chỉ số nhỏ nhất `k > i` có `PR[k] = v`" chính là "lần xuất hiện gần nhất bên phải". Quét `i` từ `m − 1` về `0`; trước khi xử lý `i`, ghi `lastR[PR[i+1]] = i + 1` và `lastC[PC[i+1]] = i + 1`. Mỗi `i` tra 4 khóa ⇒ O(1) ⇒ tổng **O(m)**.

**Insight trung gian (đơn điệu):** "chạy được ít nhất L lệnh" ⇔ trong đoạn `PR[i+1..i+L]` giá trị min/max vẫn nằm trong biên (tương tự cho cột). Điều kiện này **đơn điệu theo L** ⇒ tìm nhị phân, với min/max đoạn tra bằng sparse table O(1) ⇒ O(m log m).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sparse table min/max + tìm nhị phân (O(m log m))

Các bước:
1. Tính `PR`, `PC` (dài `m + 1`).
2. Dựng 4 sparse table: min/max của `PR` và `PC`. `table[j][x]` = min (hoặc max) của đoạn `[x, x + 2^j)`.
3. Với mỗi `i`, tìm nhị phân `L ∈ [0, m − i]` lớn nhất sao cho đoạn `[i+1, i+L]` thỏa: `startrow + minPR − PR[i] ≥ 0`, `startrow + maxPR − PR[i] ≤ n − 1`, tương tự cho cột.

In [9]:
def build_sparse(a, op):
    table = [a[:]]
    j = 1
    while (1 << j) <= len(a):
        prev = table[-1]; half = 1 << (j - 1)
        table.append([op(prev[x], prev[x + half]) for x in range(len(a) - (1 << j) + 1)])
        j += 1
    return table

def query(table, op, l, r):           # đoạn [l, r] bao gồm hai đầu
    j = (r - l + 1).bit_length() - 1
    return op(table[j][l], table[j][r - (1 << j) + 1])

t = build_sparse([5, 2, 7, 1, 9, 3], min)
print(query(t, min, 1, 4), query(t, min, 4, 5), query(build_sparse([5, 2, 7, 1, 9, 3], max), max, 0, 3))

1 3 7


In [10]:
def sparse_binary(n, startPos, s):
    m = len(s); r0, c0 = startPos
    PR, PC = prefix(s)
    mnR, mxR = build_sparse(PR, min), build_sparse(PR, max)
    mnC, mxC = build_sparse(PC, min), build_sparse(PC, max)
    def ok(i, L):                     # chạy được L lệnh đầu từ i?
        if L == 0:
            return True
        l, r = i + 1, i + L
        return (r0 + query(mnR, min, l, r) - PR[i] >= 0 and
                r0 + query(mxR, max, l, r) - PR[i] <= n - 1 and
                c0 + query(mnC, min, l, r) - PC[i] >= 0 and
                c0 + query(mxC, max, l, r) - PC[i] <= n - 1)
    ans = []
    for i in range(m):
        lo, hi = 0, m - i             # L lớn nhất với ok(i, L)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(i, mid): lo = mid
            else: hi = mid - 1
        ans.append(lo)
    return ans

for n, st, s, exp in examples:
    got = sparse_binary(n, st, s)
    print(got, "OK" if got == exp else "SAI")

[1, 5, 4, 3, 1, 0] OK
[4, 1, 0, 0] OK
[0, 0, 0, 0] OK


**Độ phức tạp cách 2:** dựng bảng O(m log m), mỗi `i` tìm nhị phân O(log m) lần tra O(1) ⇒ **O(m log m)** thời gian và bộ nhớ.

Cách này không cần Insight 2 (không nhảy cóc) — nó đúng cho cả trường hợp bước nhảy lớn hơn 1 — nên là công cụ tổng quát hơn.

### 7.2 Cách 3 — Hash map quét từ phải (O(m))

Đoạn code chia ba phần:
1. Tính `PR`, `PC`.
2. Quét `i` từ `m − 1` về `0`, trước tiên ghi `lastR[PR[i+1]] = i+1`, `lastC[PC[i+1]] = i+1`.
3. Tra 4 mục tiêu, lấy chỉ số nhỏ nhất `k*`; tính đáp án.

In [11]:
def hash_right(n, startPos, s):
    m = len(s); r0, c0 = startPos
    PR, PC = prefix(s)
    lastR, lastC = {}, {}
    ans = [0] * m
    INF = m + 1
    for i in range(m - 1, -1, -1):
        lastR[PR[i + 1]] = i + 1
        lastC[PC[i + 1]] = i + 1
        k = min(lastR.get(PR[i] - r0 - 1, INF), lastR.get(PR[i] + n - r0, INF),
                lastC.get(PC[i] - c0 - 1, INF), lastC.get(PC[i] + n - c0, INF))
        ans[i] = m - i if k == INF else k - i - 1
    return ans

for n, st, s, exp in examples:
    got = hash_right(n, st, s)
    print(got, "OK" if got == exp else "SAI")

[1, 5, 4, 3, 1, 0] OK
[4, 1, 0, 0] OK
[0, 0, 0, 0] OK


Vì sao ghi `PR[i+1]` **trước** khi tra? Ta cần `k > i`, tức `k ≥ i + 1`. Sau khi ghi, dict chứa mọi `k ∈ [i+1, m]` và với mỗi giá trị, chỉ số được giữ là **nhỏ nhất** (ghi đè lúc quét lùi). Chỉ số `k = i` không bao giờ bị tra nhầm vì `PR[i]` chỉ được ghi ở vòng `i − 1` (sau vòng hiện tại).

Mỗi mục tiêu có thể không có trong dict — `.get(..., INF)` xử lý gọn.

Code nộp LeetCode cho cách 3:

In [12]:
class SolutionHash:
    def executeInstructions(self, n: int, startPos: List[int], s: str) -> List[int]:
        m = len(s)
        r0, c0 = startPos
        PR = [0] * (m + 1); PC = [0] * (m + 1)
        for k, ch in enumerate(s):
            PR[k + 1] = PR[k] + (ch == "D") - (ch == "U")
            PC[k + 1] = PC[k] + (ch == "R") - (ch == "L")
        lastR, lastC = {}, {}
        INF = m + 1
        ans = [0] * m
        for i in range(m - 1, -1, -1):
            lastR[PR[i + 1]] = i + 1
            lastC[PC[i + 1]] = i + 1
            k = min(lastR.get(PR[i] - r0 - 1, INF), lastR.get(PR[i] + n - r0, INF),
                    lastC.get(PC[i] - c0 - 1, INF), lastC.get(PC[i] + n - c0, INF))
            ans[i] = m - i if k == INF else k - i - 1
        return ans

print(SolutionHash().executeInstructions(3, [0, 1], "RRDDLU"))

[1, 5, 4, 3, 1, 0]


Mẹo `(ch == "D") - (ch == "U")`: trong Python, `True` là 1 và `False` là 0, nên biểu thức cho +1, −1 hoặc 0.

**Độ phức tạp cách 3:** **O(m)** thời gian, O(m) bộ nhớ (hai mảng tiền tố + hai dict).

## 8. TRACE từng bước

### 8.1 Trace cách 3 trên ví dụ 1

`n = 3`, `(r0, c0) = (0, 1)`, `s = "RRDDLU"`. Mảng tiền tố:

| k | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| lệnh trước k | — | R | R | D | D | L | U |
| PR[k] | 0 | 0 | 0 | 1 | 2 | 2 | 1 |
| PC[k] | 0 | 1 | 2 | 2 | 2 | 1 | 1 |

Mục tiêu với `i`: hàng `PR[i] − 1` (lên quá) và `PR[i] + 3` (xuống quá); cột `PC[i] − 2` (trái quá) và `PC[i] + 2` (phải quá).

Ví dụ `i = 0`: mục tiêu cột phải = `0 + 2 = 2`, lần đầu `PC[k] = 2` với `k > 0` là `k = 2` ⇒ `answer[0] = 2 − 0 − 1 = 1`. ✔

In [13]:
def trace_hash(n, startPos, s):
    m = len(s); r0, c0 = startPos
    PR, PC = prefix(s)
    print("PR =", PR); print("PC =", PC)
    lastR, lastC = {}, {}; INF = m + 1; ans = [0] * m
    for i in range(m - 1, -1, -1):
        lastR[PR[i + 1]] = i + 1; lastC[PC[i + 1]] = i + 1
        tg = {"lên": (lastR, PR[i] - r0 - 1), "xuống": (lastR, PR[i] + n - r0),
              "trái": (lastC, PC[i] - c0 - 1), "phải": (lastC, PC[i] + n - c0)}
        hits = {name: d.get(v, INF) for name, (d, v) in tg.items()}
        k = min(hits.values())
        ans[i] = m - i if k == INF else k - i - 1
        shown = {nm: (v, ("-" if hits[nm] == INF else hits[nm])) for nm, (_, v) in tg.items()}
        print(f"i={i}: lastR={lastR} lastC={lastC}")
        print(f"      mục tiêu(giá trị, k) = {shown} -> k*={'-' if k == INF else k}, ans[{i}]={ans[i]}")
    return ans

trace_hash(3, [0, 1], "RRDDLU")

PR = [0, 0, 0, 1, 2, 2, 1]
PC = [0, 1, 2, 2, 2, 1, 1]
i=5: lastR={1: 6} lastC={1: 6}
      mục tiêu(giá trị, k) = {'lên': (1, 6), 'xuống': (5, '-'), 'trái': (-1, '-'), 'phải': (3, '-')} -> k*=6, ans[5]=0
i=4: lastR={1: 6, 2: 5} lastC={1: 5}
      mục tiêu(giá trị, k) = {'lên': (1, 6), 'xuống': (5, '-'), 'trái': (0, '-'), 'phải': (4, '-')} -> k*=6, ans[4]=1
i=3: lastR={1: 6, 2: 4} lastC={1: 5, 2: 4}
      mục tiêu(giá trị, k) = {'lên': (0, '-'), 'xuống': (4, '-'), 'trái': (0, '-'), 'phải': (4, '-')} -> k*=-, ans[3]=3
i=2: lastR={1: 3, 2: 4} lastC={1: 5, 2: 3}
      mục tiêu(giá trị, k) = {'lên': (-1, '-'), 'xuống': (3, '-'), 'trái': (0, '-'), 'phải': (4, '-')} -> k*=-, ans[2]=4
i=1: lastR={1: 3, 2: 4, 0: 2} lastC={1: 5, 2: 2}
      mục tiêu(giá trị, k) = {'lên': (-1, '-'), 'xuống': (3, '-'), 'trái': (-1, '-'), 'phải': (3, '-')} -> k*=-, ans[1]=5
i=0: lastR={1: 3, 2: 4, 0: 1} lastC={1: 1, 2: 2}
      mục tiêu(giá trị, k) = {'lên': (-1, '-'), 'xuống': (3, '-'), 'trái': (-2, '-'), 'phải': 

[1, 5, 4, 3, 1, 0]

### 8.2 Trace cách 1 (mô phỏng) cho một hậu tố cụ thể

In vị trí sau mỗi lệnh khi bắt đầu từ `i = 0` ở ví dụ 2 (`n = 2`, start `(1, 1)`, `"LURD"`): robot đi một vòng hình vuông và chạy hết 4 lệnh.

In [14]:
def trace_sim(n, startPos, s, i):
    r, c = startPos
    print(f"bắt đầu ở {(r, c)}, chạy từ lệnh {i}")
    for j in range(i, len(s)):
        dr, dc = DIRS[s[j]]
        nr, nc = r + dr, c + dc
        ok = 0 <= nr < n and 0 <= nc < n
        print(f"  lệnh {j} '{s[j]}': {(r, c)} -> {(nr, nc)} {'OK' if ok else 'RA NGOÀI, dừng'}")
        if not ok: return j - i
        r, c = nr, nc
    return len(s) - i

print("answer =", trace_sim(2, [1, 1], "LURD", 0))
print("answer =", trace_sim(2, [1, 1], "LURD", 1))

bắt đầu ở (1, 1), chạy từ lệnh 0
  lệnh 0 'L': (1, 1) -> (1, 0) OK
  lệnh 1 'U': (1, 0) -> (0, 0) OK
  lệnh 2 'R': (0, 0) -> (0, 1) OK
  lệnh 3 'D': (0, 1) -> (1, 1) OK
answer = 4
bắt đầu ở (1, 1), chạy từ lệnh 1
  lệnh 1 'U': (1, 1) -> (0, 1) OK
  lệnh 2 'R': (0, 1) -> (0, 2) RA NGOÀI, dừng
answer = 1


### 8.3 Trace cách 2 (tìm nhị phân) cho `i = 1` ở ví dụ 1

Ta in từng `mid` được thử và kết quả `ok(i, mid)`.

In [15]:
def trace_binary(n, startPos, s, i):
    m = len(s); r0, c0 = startPos; PR, PC = prefix(s)
    def ok(L):
        if L == 0: return True
        seg_r = PR[i + 1:i + L + 1]; seg_c = PC[i + 1:i + L + 1]
        return (r0 + min(seg_r) - PR[i] >= 0 and r0 + max(seg_r) - PR[i] <= n - 1 and
                c0 + min(seg_c) - PC[i] >= 0 and c0 + max(seg_c) - PC[i] <= n - 1)
    lo, hi = 0, m - i
    while lo < hi:
        mid = (lo + hi + 1) // 2
        res = ok(mid)
        print(f"  lo={lo} hi={hi} thử mid={mid}: {'chạy được' if res else 'ra ngoài'}")
        if res: lo = mid
        else: hi = mid - 1
    print(f"  -> answer[{i}] = {lo}")

trace_binary(3, [0, 1], "RRDDLU", 0)
trace_binary(3, [0, 1], "RRDDLU", 4)

  lo=0 hi=6 thử mid=3: ra ngoài
  lo=0 hi=2 thử mid=1: chạy được
  lo=1 hi=2 thử mid=2: ra ngoài
  -> answer[0] = 1
  lo=0 hi=2 thử mid=1: chạy được
  lo=1 hi=2 thử mid=2: ra ngoài
  -> answer[4] = 1


## 9. Code nộp LeetCode

Với `m ≤ 500`, mô phỏng là đủ và dễ đọc nhất. Ta nộp bản mô phỏng; bản O(m) (`SolutionHash`) ở mục 7.2 cũng nộp được.

In [16]:
from typing import List

class Solution:
    def executeInstructions(self, n: int, startPos: List[int], s: str) -> List[int]:
        moves = {"L": (0, -1), "R": (0, 1), "U": (-1, 0), "D": (1, 0)}
        m = len(s)
        ans = []
        for i in range(m):
            r, c = startPos
            cnt = 0
            for j in range(i, m):
                dr, dc = moves[s[j]]
                r, c = r + dr, c + dc
                if not (0 <= r < n and 0 <= c < n):
                    break
                cnt += 1
            ans.append(cnt)
        return ans

In [17]:
sol = Solution()
for n, st, s, exp in examples:
    got = sol.executeInstructions(n, st, s)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

[1, 5, 4, 3, 1, 0] | kỳ vọng [1, 5, 4, 3, 1, 0] | OK
[4, 1, 0, 0] | kỳ vọng [4, 1, 0, 0] | OK
[0, 0, 0, 0] | kỳ vọng [0, 0, 0, 0] | OK


Lưu ý: bản nộp cập nhật `r, c` **trước** rồi mới kiểm tra — vẫn đúng vì nếu ra ngoài ta `break` ngay và không dùng `r, c` nữa; và `r, c = startPos` được đặt lại ở đầu mỗi `i`. Unpack `r, c = startPos` tạo hai biến mới, không sửa list `startPos`.

## 10. Vì sao thuật toán đúng

**Cách 1** là định nghĩa của đề viết thành code: mỗi `i` chạy độc lập từ `startPos`, dừng đúng lúc lệnh kế tiếp ra ngoài.

**Cách 3.** Cố định `i`. Gọi `pos(k) = start + P[k] − P[i]` với `k ≥ i` (`pos(i) = start` nằm trong lưới).
- Robot thực hiện được lệnh `k − 1` (`k > i`) ⇔ `pos(i+1), …, pos(k)` đều trong lưới.
- Toạ độ hàng của `pos` thay đổi tối đa 1 mỗi bước, bắt đầu trong `[0, n−1]`. Theo "định lý giá trị trung gian rời rạc", lần đầu tiên nó ra khỏi `[0, n−1]` thì nó bằng đúng −1 hoặc n. Tương tự cho cột.
- Do đó chỉ số thất bại đầu tiên `k*` = min các `k > i` mà hàng ∈ {−1, n} hoặc cột ∈ {−1, n} — chính là min 4 tra cứu. Trước `k*`, mọi vị trí trong lưới (nếu không thì đã có thất bại sớm hơn, mâu thuẫn tính nhỏ nhất).
- Số lệnh đã chạy = `(k* − 1) − i` (các lệnh `i..k*−2`); nếu không có `k*` thì chạy hết `m − i` lệnh.

**Cách 2.** `ok(i, L)` đúng ⇔ mọi vị trí `pos(i+1..i+L)` trong lưới ⇔ min/max từng toạ độ trong đoạn nằm trong biên. `ok(i, L)` ⇒ `ok(i, L − 1)` (đơn điệu) ⇒ tìm nhị phân tìm đúng L lớn nhất.

## 11. Phân tích Time / Space Complexity

`m = len(s)`, `n` là cạnh lưới (không ảnh hưởng độ phức tạp ngoài việc quyết định robot ra ngoài sớm hay muộn).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng | O(m²) worst | O(1) thêm | worst khi robot đi qua lại không bao giờ ra ngoài; nếu lưới nhỏ thì thường dừng sớm |
| 2. Sparse + nhị phân | O(m log m) | O(m log m) | không cần tính chất ±1 |
| 3. Hash quét phải | O(m) | O(m) | dùng tính chất ±1 |

Đo trên `m` lớn hơn ràng buộc để thấy khác biệt:

In [18]:
rng = random.Random(3)
for m in [500, 3000]:
    s = "".join(rng.choice("LRUD") for _ in range(m)); n = 60; st = [30, 30]
    res = {}
    for name, f in [("simulate", simulate_all), ("sparse", sparse_binary), ("hash", hash_right)]:
        t = time.perf_counter(); res[name] = f(n, st, s); res[name + "_t"] = time.perf_counter() - t
    assert res["simulate"] == res["sparse"] == res["hash"]
    print(f"m={m}: simulate {res['simulate_t']*1000:.0f} ms, sparse {res['sparse_t']*1000:.0f} ms, hash {res['hash_t']*1000:.1f} ms")

m=500: simulate 10 ms, sparse 3 ms, hash 0.2 ms


m=3000: simulate 258 ms, sparse 23 ms, hash 1.1 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {
    "simulate": simulate_all, "sparse": sparse_binary, "hash": hash_right,
    "SolutionHash": lambda n, p, s: SolutionHash().executeInstructions(n, p, s),
    "Solution": lambda n, p, s: Solution().executeInstructions(n, p, s),
}
edge = [e for e in examples] + [
    (1, [0, 0], "L", [0]),
    (5, [2, 2], "R", [1]),
    (2, [0, 0], "RRRR", [1, 1, 1, 1]),
    (2, [0, 0], "RLRL", [4, 0, 2, 0]),      # bắt đầu bằng "L" từ (0,0) là ra ngoài ngay
    (3, [2, 2], "DDDD", [0, 0, 0, 0]),
    (500, [0, 0], "D" * 10, [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]),
    (3, [1, 1], "UDUDLRLR", [8, 7, 6, 5, 4, 3, 2, 1]),
]
for n, st, s, exp in edge:
    for name, f in ALL.items():
        got = f(n, list(st), s)
        assert got == exp, (name, n, st, s, got, exp)
print("Edge case OK:", len(edge))

Edge case OK: 10


### 12.2 Đối chiếu ngẫu nhiên (seed cố định, đúng ràng buộc)

Sinh `n ∈ [1, 8]` (lưới nhỏ để robot hay ra ngoài) và `n` lớn; `m ∈ [1, 60]`. Đồng thời kiểm tra `startPos` không bị sửa.

In [20]:
rng = random.Random(2120)
for _ in range(3000):
    n = rng.choice([1, 2, 3, 4, 5, 8, 50])
    st = [rng.randrange(n), rng.randrange(n)]
    m = rng.randint(1, 60)
    s = "".join(rng.choice("LRUD") for _ in range(m))
    ref = simulate_all(n, st, s)
    for name, f in ALL.items():
        st_copy = list(st)
        assert f(n, st_copy, s) == ref, (name, n, st, s)
        assert st_copy == st, (name, "đã sửa startPos")
print("3000 case ngẫu nhiên OK")

3000 case ngẫu nhiên OK


### 12.3 Kích thước lớn nhất (`n = m = 500`)

In [21]:
rng = random.Random(500)
s = "".join(rng.choice("LRUD") for _ in range(500))
t = time.perf_counter(); a1 = Solution().executeInstructions(500, [250, 250], s); t1 = time.perf_counter() - t
a2 = SolutionHash().executeInstructions(500, [250, 250], s)
assert a1 == a2 and a1 == [500 - i for i in range(500)]   # lưới rất lớn: không bao giờ ra ngoài
s2 = "".join(rng.choice("LRUD") for _ in range(500))
assert Solution().executeInstructions(3, [1, 1], s2) == SolutionHash().executeInstructions(3, [1, 1], s2)
print(f"OK, mô phỏng worst-case mất {t1*1000:.1f} ms")

OK, mô phỏng worst-case mất 11.5 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không đặt lại vị trí robot cho mỗi `i`

In [22]:
def wrong_no_reset(n, startPos, s):
    m = len(s); ans = []
    r, c = startPos                    # SAI: đặt một lần ngoài vòng for i
    for i in range(m):
        cnt = 0
        for j in range(i, m):
            dr, dc = DIRS[s[j]]
            if 0 <= r + dr < n and 0 <= c + dc < n:
                r, c, cnt = r + dr, c + dc, cnt + 1
            else:
                break
        ans.append(cnt)
    return ans

print("bản sai :", wrong_no_reset(3, [0, 1], "RRDDLU"))
print("bản đúng:", Solution().executeInstructions(3, [0, 1], "RRDDLU"))

bản sai : [1, 0, 4, 3, 0, 1]
bản đúng: [1, 5, 4, 3, 1, 0]


**Vì sao sai:** lần chạy từ `i = 1` bắt đầu ở vị trí mà lần chạy `i = 0` dừng lại (`(0, 2)`), không phải `startPos`. **Sửa:** `r, c = startPos` **bên trong** vòng `for i`.

### Lỗi 2 — Nhầm chiều `U`/`D` (coi `U` là hàng +1)

In [23]:
WRONG_DIRS = {"L": (0, -1), "R": (0, 1), "U": (1, 0), "D": (-1, 0)}   # SAI
def wrong_dirs(n, startPos, s):
    ans = []
    for i in range(len(s)):
        r, c = startPos; cnt = 0
        for ch in s[i:]:
            dr, dc = WRONG_DIRS[ch]
            r, c = r + dr, c + dc
            if not (0 <= r < n and 0 <= c < n): break
            cnt += 1
        ans.append(cnt)
    return ans

print("bản sai :", wrong_dirs(3, [0, 1], "RRDDLU"))
print("bản đúng:", Solution().executeInstructions(3, [0, 1], "RRDDLU"))

bản sai : [1, 1, 0, 0, 2, 1]
bản đúng: [1, 5, 4, 3, 1, 0]


**Vì sao sai:** đề quy ước `(0,0)` ở **trái-trên**, nên đi xuống làm **hàng tăng**. Hệ toạ độ toán học (y hướng lên) khác hệ toạ độ ma trận. **Sửa:** `'U': (-1, 0)`, `'D': (1, 0)`.

### Lỗi 3 — Sửa trực tiếp `startPos` (alias list)

In [24]:
def wrong_alias(n, startPos, s):
    ans = []
    for i in range(len(s)):
        pos = startPos                     # SAI: pos và startPos là CÙNG một list
        cnt = 0
        for ch in s[i:]:
            dr, dc = DIRS[ch]
            if 0 <= pos[0] + dr < n and 0 <= pos[1] + dc < n:
                pos[0] += dr; pos[1] += dc; cnt += 1
            else:
                break
        ans.append(cnt)
    return ans

st = [0, 1]
print("bản sai :", wrong_alias(3, st, "RRDDLU"), "| startPos sau khi gọi:", st)
print("bản đúng:", Solution().executeInstructions(3, [0, 1], "RRDDLU"))

bản sai : [1, 0, 4, 3, 0, 1] | startPos sau khi gọi: [0, 0]
bản đúng: [1, 5, 4, 3, 1, 0]


**Vì sao sai:** `pos = startPos` không sao chép; mọi thay đổi trên `pos` làm hỏng `startPos` cho các lần chạy sau. **Sửa:** `r, c = startPos` (hai số nguyên bất biến) hoặc `pos = list(startPos)`.

### Lỗi 4 — Cách 3: ghi `PR[i]` thay vì `PR[i+1]` vào dict (cho phép `k = i`)

In [25]:
def wrong_hash_index(n, startPos, s):
    m = len(s); r0, c0 = startPos; PR, PC = prefix(s)
    lastR, lastC = {}, {}; INF = m + 1; ans = [0] * m
    for i in range(m - 1, -1, -1):
        lastR[PR[i]] = i; lastC[PC[i]] = i            # SAI: đưa k = i vào
        k = min(lastR.get(PR[i] - r0 - 1, INF), lastR.get(PR[i] + n - r0, INF),
                lastC.get(PC[i] - c0 - 1, INF), lastC.get(PC[i] + n - c0, INF))
        ans[i] = m - i if k == INF else k - i - 1
    return ans

print("bản sai :", wrong_hash_index(3, [0, 1], "RRDDLU"))
print("bản đúng:", hash_right(3, [0, 1], "RRDDLU"))

bản sai : [1, 5, 4, 3, 2, 1]
bản đúng: [1, 5, 4, 3, 1, 0]


**Vì sao sai:** bản sai ghi `PR[i]` ở vòng `i`, nên vị trí `k = i + 1` (lệnh `i` thất bại ngay) chỉ được ghi **sau** khi đã tra cho `i` — bỏ sót đúng trường hợp quan trọng nhất (dòng `i = 0` ra 2 thay vì 1). **Sửa:** ở vòng `i`, ghi `PR[i+1]`, `PC[i+1]` rồi mới tra.

### Lỗi 5 — Kiểm tra biên `<= n` thay vì `< n`

In [26]:
def wrong_bound(n, startPos, s):
    ans = []
    for i in range(len(s)):
        r, c = startPos; cnt = 0
        for ch in s[i:]:
            dr, dc = DIRS[ch]; r, c = r + dr, c + dc
            if not (0 <= r <= n and 0 <= c <= n): break     # SAI: cho phép r == n
            cnt += 1
        ans.append(cnt)
    return ans

print("bản sai :", wrong_bound(3, [0, 1], "RRDDLU"))
print("bản đúng:", Solution().executeInstructions(3, [0, 1], "RRDDLU"))

bản sai : [6, 5, 4, 3, 1, 0]
bản đúng: [1, 5, 4, 3, 1, 0]


**Sửa:** chỉ số hợp lệ là `0..n−1` ⇒ điều kiện `0 <= r < n`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Đọc ràng buộc trước:** `m ≤ 500` ⇒ O(m²) = 2,5·10⁵ thao tác ⇒ cứ mô phỏng. Đừng tối ưu sớm trong phòng thi.
- Di chuyển trên lưới: luôn dùng **bảng hướng** `dict`/list, và chú ý quy ước hàng/cột của đề.
- "Mọi hậu tố / mọi điểm bắt đầu" + "vị trí = hiệu tiền tố" ⇒ nghĩ tới **prefix sum** và **tra cứu lần xuất hiện gần nhất** (hash map quét ngược) — pattern giống "subarray sum equals k" (LC 560).
- Bước nhảy ±1 ⇒ không thể nhảy qua biên ⇒ "ra ngoài lần đầu" = "chạm đúng giá trị biên lần đầu".
- Khi điều kiện đơn điệu theo độ dài ⇒ tìm nhị phân + truy vấn min/max đoạn (sparse table) là phương án dự phòng tổng quát.
- Bài cùng họ: 874 Walking Robot Simulation, 1496 Path Crossing, 657 Robot Return to Origin, 2056.

## 15. Biến thể của bài

### 15.1 Biến thể A — Lưới chữ nhật `rows × cols` và có **ô cấm**

Robot dừng khi lệnh kế tiếp ra ngoài **hoặc** đi vào ô cấm. Tính chất "không nhảy cóc" vẫn còn nhưng ô cấm nằm rải rác nên cách 3 không áp dụng trực tiếp; mô phỏng O(m²) vẫn đơn giản và đúng.

In [27]:
def execute_with_walls(rows, cols, walls, startPos, s):
    walls = set(map(tuple, walls)); ans = []
    for i in range(len(s)):
        r, c = startPos; cnt = 0
        for ch in s[i:]:
            dr, dc = DIRS[ch]; nr, nc = r + dr, c + dc
            if not (0 <= nr < rows and 0 <= nc < cols) or (nr, nc) in walls: break
            r, c, cnt = nr, nc, cnt + 1
        ans.append(cnt)
    return ans

print(execute_with_walls(3, 4, [[1, 2]], [0, 1], "RRDDLU"))
print(execute_with_walls(3, 3, [], [0, 1], "RRDDLU"))   # không ô cấm = bài gốc

[5, 1, 4, 3, 1, 0]
[1, 5, 4, 3, 1, 0]


### 15.2 Biến thể B — Lệnh có **bước nhảy k ô** (ví dụ `"R3"`)

Khi bước nhảy > 1, robot có thể nhảy qua biên (`n − 1 → n + 2`), nên cách 3 (tra đúng giá trị) **sai**; nhưng cách 2 (sparse table + nhị phân trên min/max) vẫn đúng nguyên vẹn. Ta cài bằng cách đổi hàm tiền tố.

In [28]:
def execute_with_jumps(n, startPos, cmds):
    # cmds: list các cặp (hướng, độ dài)
    m = len(cmds); r0, c0 = startPos
    PR, PC = [0], [0]
    for d, k in cmds:
        dr, dc = DIRS[d]; PR.append(PR[-1] + dr * k); PC.append(PC[-1] + dc * k)
    mnR, mxR, mnC, mxC = build_sparse(PR, min), build_sparse(PR, max), build_sparse(PC, min), build_sparse(PC, max)
    def ok(i, L):
        if L == 0: return True
        l, r = i + 1, i + L
        return (r0 + query(mnR, min, l, r) - PR[i] >= 0 and r0 + query(mxR, max, l, r) - PR[i] <= n - 1 and
                c0 + query(mnC, min, l, r) - PC[i] >= 0 and c0 + query(mxC, max, l, r) - PC[i] <= n - 1)
    ans = []
    for i in range(m):
        lo, hi = 0, m - i
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(i, mid): lo = mid
            else: hi = mid - 1
        ans.append(lo)
    return ans

def jumps_brute(n, startPos, cmds):
    ans = []
    for i in range(len(cmds)):
        r, c = startPos; cnt = 0
        for d, k in cmds[i:]:
            dr, dc = DIRS[d]; r, c = r + dr * k, c + dc * k
            if not (0 <= r < n and 0 <= c < n): break
            cnt += 1
        ans.append(cnt)
    return ans

cmds = [("R", 2), ("D", 3), ("L", 1), ("U", 2), ("R", 1)]
print(execute_with_jumps(5, [0, 0], cmds), jumps_brute(5, [0, 0], cmds))
rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 7); st = [rng.randrange(n), rng.randrange(n)]
    cmds = [(rng.choice("LRUD"), rng.randint(1, 3)) for _ in range(rng.randint(1, 25))]
    assert execute_with_jumps(n, st, cmds) == jumps_brute(n, st, cmds)
print("biến thể bước nhảy: khớp mô phỏng")

[5, 1, 0, 0, 1] [5, 1, 0, 0, 1]
biến thể bước nhảy: khớp mô phỏng


### 15.3 Biến thể C — Chỉ hỏi **một** điểm bắt đầu nhưng nhiều truy vấn `startPos`

Nếu `s` cố định và có `q` truy vấn `(startPos, i)`, dựng sparse table một lần rồi mỗi truy vấn tìm nhị phân O(log m) ⇒ tổng O((m + q) log m). Đây là lý do cách 2 đáng học: nó tách phần "tiền xử lý chuỗi lệnh" khỏi "vị trí xuất phát".

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | chạy lại từng hậu tố | O(m²) | O(1) | **ràng buộc nhỏ (m ≤ 500)** — mặc định ở bài này |
| 2. Sparse table + nhị phân | min/max tiền tố trong đoạn, đơn điệu theo độ dài | O(m log m) | O(m log m) | bước nhảy tùy ý, nhiều truy vấn |
| 3. Hash map quét phải | lần đầu chạm giá trị biên của tiền tố | O(m) | O(m) | m lớn, bước ±1 |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 2, startPos = [0, 0], s = "DRUL"` cho kết quả gì?

**Bài 2 (tính tay):** Với `s = "RRDDLU"`, tính `PR[6] − PR[2]` và `PC[6] − PC[2]`. Robot bắt đầu từ lệnh 2 kết thúc ở đâu (start `(0,1)`)?

**Bài 3 (điền TODO):** Viết `bt3_todo(n, startPos, s)` trả về **vị trí cuối cùng** của robot cho mỗi điểm bắt đầu `i` (list các cặp `(r, c)`), dùng mô phỏng.

**Bài 4 (tự viết lại):** Viết `bt4_todo(n, startPos, s)` cài lại cách 3 (hash map quét phải) mà không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(n, startPos, s)` trả về chỉ số `i` cho **nhiều lệnh thực hiện được nhất** (nếu hòa lấy `i` nhỏ nhất) — yêu cầu O(m) bằng cách tận dụng `bt4`/cách 3.

In [29]:
def bt3_todo(n, startPos, s):
    # TODO: list vị trí cuối cho mỗi i
    pass

def bt4_todo(n, startPos, s):
    # TODO: cách 3 - hash map quét từ phải
    pass

def bt5_todo(n, startPos, s):
    # TODO: chỉ số i có nhiều lệnh thực hiện nhất
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: vẽ lưới 2×2. Mức 2: từ `(0,0)`: D→(1,0), R→(1,1), U→(0,1), L→(0,0) — hình vuông. Mức 3: `i=0` chạy hết 4; `i=1` R→(0,1), U ra ngoài ⇒ 1; `i=2` U ra ngoài ⇒ 0; `i=3` L ra ngoài ⇒ 0.

**Bài 2.** Mức 1: dùng bảng PR/PC ở mục 8.1. Mức 2: `PR[6] − PR[2] = 1 − 0`, `PC[6] − PC[2] = 1 − 2`. Mức 3: vị trí `(0 + 1, 1 − 1) = (1, 0)` — khớp mục 4.

**Bài 3.** Mức 1: giống `Solution`, nhưng thay vì `cnt` hãy trả `(r, c)` cuối cùng *trong lưới*. Mức 2: chỉ cập nhật `r, c` khi bước mới hợp lệ. Mức 3: dùng cấu trúc của `simulate_all`, append `(r, c)` thay vì `cnt`.

**Bài 4.** Mức 1: tiền tố `PR`, `PC`. Mức 2: 4 mục tiêu `PR[i]−r0−1`, `PR[i]+n−r0`, `PC[i]−c0−1`, `PC[i]+n−c0`. Mức 3: ghi `PR[i+1]` trước khi tra; `k*` nhỏ nhất ⇒ `k*−i−1`.

**Bài 5.** Mức 1: có mảng `answer` rồi thì chỉ cần argmax. Mức 2: `max(range(m), key=lambda i: (ans[i], -i))`. Mức 3: hoặc duyệt tay, chỉ cập nhật khi `ans[i] > best` (giữ `i` nhỏ nhất khi hòa).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[4, 1, 0, 0]`. **Bài 2:** `PR[6]−PR[2] = 1`, `PC[6]−PC[2] = −1` ⇒ kết thúc ở `(1, 0)`.

In [30]:
assert Solution().executeInstructions(2, [0, 0], "DRUL") == [4, 1, 0, 0]
PR, PC = prefix("RRDDLU")
assert (PR[6] - PR[2], PC[6] - PC[2]) == (1, -1)

def bt3(n, startPos, s):
    res = []
    for i in range(len(s)):
        r, c = startPos
        for ch in s[i:]:
            dr, dc = DIRS[ch]
            if not (0 <= r + dr < n and 0 <= c + dc < n): break
            r, c = r + dr, c + dc
        res.append((r, c))
    return res

def bt4(n, startPos, s):
    m = len(s); r0, c0 = startPos
    PR = [0] * (m + 1); PC = [0] * (m + 1)
    for k, ch in enumerate(s):
        dr, dc = DIRS[ch]; PR[k + 1] = PR[k] + dr; PC[k + 1] = PC[k] + dc
    lr, lc = {}, {}; ans = [0] * m; INF = m + 1
    for i in range(m - 1, -1, -1):
        lr[PR[i + 1]] = i + 1; lc[PC[i + 1]] = i + 1
        k = min(lr.get(PR[i] - r0 - 1, INF), lr.get(PR[i] + n - r0, INF),
                lc.get(PC[i] - c0 - 1, INF), lc.get(PC[i] + n - c0, INF))
        ans[i] = m - i if k == INF else k - i - 1
    return ans

def bt5(n, startPos, s):
    ans = bt4(n, startPos, s)
    best = 0
    for i in range(1, len(ans)):
        if ans[i] > ans[best]: best = i
    return best

assert bt3(3, [0, 1], "RRDDLU") == [(0, 2), (1, 1), (1, 0), (0, 0), (0, 0), (0, 1)]
assert bt5(3, [0, 1], "RRDDLU") == 1 and bt5(1, [0, 0], "LRUD") == 0
rng = random.Random(1920)
for _ in range(1500):
    n = rng.randint(1, 6); st = [rng.randrange(n), rng.randrange(n)]
    s = "".join(rng.choice("LRUD") for _ in range(rng.randint(1, 40)))
    ref = simulate_all(n, st, s)
    assert bt4(n, st, s) == ref
    assert bt5(n, st, s) == ref.index(max(ref))
    # vị trí cuối của bt3 phải khớp với tiền tố tại bước dừng
    PRs, PCs = prefix(s)
    for i, (r, c) in enumerate(bt3(n, st, s)):
        k = i + ref[i]
        assert (r, c) == (st[0] + PRs[k] - PRs[i], st[1] + PCs[k] - PCs[i])
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Mô phỏng (m nhỏ)
moves = {'L': (0,-1), 'R': (0,1), 'U': (-1,0), 'D': (1,0)}
for i in range(m):
    r, c = startPos; cnt = 0
    for ch in s[i:]:
        dr, dc = moves[ch]; r += dr; c += dc
        if not (0 <= r < n and 0 <= c < n): break
        cnt += 1
    ans.append(cnt)

# O(m): tiền tố + lần chạm biên đầu tiên (quét phải, ghi P[i+1] trước khi tra)
```

**Checklist:**
- [ ] `U` là hàng −1, `D` là hàng +1.
- [ ] Đặt lại `r, c = startPos` cho **mỗi** `i`; không alias list `startPos`.
- [ ] Biên `0 <= x < n` (không phải `<= n`).
- [ ] Đếm lệnh **đã thực hiện**, không đếm lệnh làm ra ngoài.
- [ ] Đã thử `n = 1` (mọi lệnh đều ra ngoài) và lưới lớn (chạy hết).